In [1]:
"""
Lu Section 4 simulations
"""

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time
import pickle

from dataclasses import dataclass, asdict

In [2]:
project_root = os.path.abspath(os.getcwd())
sys.path.insert(0, project_root)


from BayesianSparseDeepHalo.LuSparseRandomLogit import BayesianSparseRandomLogit
from BayesianSparseDeepHalo.GenerateData import generate_data_tf
from BayesianSparseDeepHalo.BLP import blp_estimator_fast

In [3]:
# =====================================
# PARAMETERS
# =====================================


@dataclass
class SimParams:
    """Data Generation Parameters"""
    T: int = 100  # Number of Markets
    J: int = 15   # Number of Products per Market
    Nt: int = 1000  # Consumers per Market
    D: int = 2    # Number of features
    beta_mean: np.ndarray = None  # Mean of beta (D,)
    sigma_beta: np.ndarray = None  # Std of beta (D,)
    xi_bar: float = -1.0
    seed: int = 123

    def __post_init__(self):
        if self.beta_mean is None:
            self.beta_mean = np.array([-1.0, 0.5])
        if self.sigma_beta is None:
            self.sigma_beta = np.array([1.5, 0.0])  # Price random, w fixed (0 means fixed)

@dataclass
class MCMCParams:
    """Hyperparameters for the Bayesian Algorithm"""
    R0: int = 200  # Fixed simulation draws
    G: int = 10000  # Total iterations
    burn: int = 5000  # Burn-in samples

    # Masking: which beta components are random (1) vs fixed (0)
    random_coef_mask: np.ndarray = None

    # Priors
    tau0: float = 1e-3
    tau1: float = 1.0
    a_phi: float = 1.0
    b_phi: float = 1.0
    V_beta: float = 10.0
    V_xibar: float = 10.0
    V_r: float = 0.5

    # Initial Steps
    step_beta: float = 0.5
    step_r: float = 0.5
    step_xibar: float = 0.5
    step_eta: float = 0.5
    step_phi: float = 0.5

    # Target acceptance
    target_accept: float = 0.35

    # Feature dimension
    D: int = 2

    def __post_init__(self):
        if self.random_coef_mask is None:
            # Default: first coefficient is random, rest are fixed
            self.random_coef_mask = np.zeros(self.D)
            self.random_coef_mask[0] = 1.0



In [4]:
# =====================================
# MCMC and BLP RUNNERS
# =====================================

def run_choice_learn_mcmc(choice_dataset, sim_params, mcmc_params, **kwargs):
    model = BayesianSparseRandomLogit(sim_params=sim_params, mcmc_params=mcmc_params, **kwargs)
    losses = model.fit(choice_dataset, store_samples=True)
    return model.samples_, losses, model
    

def _blp_estimate_one(data, true, N_draw=500, seed=111):
    u_cost = true["u_cost"]
    out1 = blp_estimator_fast(data, u_cost, iv_type=1, N_draw=N_draw, seed=seed)
    out2 = blp_estimator_fast(data, u_cost, iv_type=2, N_draw=N_draw, seed=seed)
    out3 = blp_estimator_fast(data, u_cost, iv_type=3, N_draw=N_draw, seed=seed)

    return out1, out2, out3


In [5]:
# =====================================
# METRICS & UTILS
# =====================================

def _compute_metrics_one_rep(draws: dict, true: dict, sim: SimParams):
    # Lu2025 posterior-mean estimates
    beta_draws = draws["beta_bar"]          # (keep, D)
    r_draws = draws["r_vec"]                # (keep, n_random)
    xibar_draws = draws["xi_bar"]           # (keep, T)
    eta_draws = draws["eta"]                # (keep, T, J)
    gamma_draws = draws["gamma"]            # (keep, T, J)

    beta_hat = beta_draws.mean(axis=0)
    sigma_hat_vec = np.exp(r_draws).mean(axis=0)
    xibar_hat = xibar_draws.mean(axis=0)

    # xi_{t,j} = xi_bar_t + eta_{t,j}
    xi_draws = xibar_draws[:, :, None] + eta_draws
    xi_hat = xi_draws.mean(axis=0)         # (T, J)

    # posterior inclusion probability
    gamma_prob = gamma_draws.mean(axis=0)  # (T, J)

    # condition on TRUE eta_star being zero vs nonzero
    eta_true = true["eta_star"]
    mask_nz = (np.abs(eta_true) > 1e-12)
    mask_z = ~mask_nz

    # Store generic metrics
    metrics = {
        "xibar_hat_mean": float(xibar_hat.mean()),
        "xi_hat": xi_hat,
        "xi_true": true["xi_star"],
        "pr_gamma_1_eta_nz": float(gamma_prob[mask_nz].mean()) if mask_nz.any() else np.nan,
        "pr_gamma_1_eta_z": float(gamma_prob[mask_z].mean()) if mask_z.any() else np.nan,
    }

    # Add betas
    for d in range(sim.D):
        metrics[f"beta_{d}_hat"] = float(beta_hat[d])

    # Add sigmas (assuming 1st random coef matches sigma_p for D=2 compat)
    if len(sigma_hat_vec) > 0:
        metrics["sigma_hat"] = float(sigma_hat_vec[0])
    else:
        metrics["sigma_hat"] = 0.0

    return metrics

def _summarize_across_reps(rep_metrics: list, sim: SimParams):
    # Aggregation
    xibar_hats = np.array([met["xibar_hat_mean"] for met in rep_metrics])

    out = {
        "bias_xibar": float(xibar_hats.mean() - sim.xi_bar),
        "sd_xibar": float(xibar_hats.std(ddof=1)),
        "pr_gamma_1_eta_nz": float(np.nanmean([met["pr_gamma_1_eta_nz"] for met in rep_metrics])),
        "pr_gamma_1_eta_z": float(np.nanmean([met["pr_gamma_1_eta_z"] for met in rep_metrics])),
    }

    # Beta stats
    for d in range(sim.D):
        b_hats = np.array([met[f"beta_{d}_hat"] for met in rep_metrics])
        out[f"bias_beta_{d}"] = float(b_hats.mean() - sim.beta_mean[d])
        out[f"sd_beta_{d}"] = float(b_hats.std(ddof=1))

    # Sigma stats (only 1st one for table compat)
    # Assuming sim.sigma_beta[0] is the target
    s_hats = np.array([met["sigma_hat"] for met in rep_metrics])
    target_sigma = sim.sigma_beta[0] if sim.sigma_beta is not None else 0.0
    out["bias_sigma"] = float(s_hats.mean() - target_sigma)
    out["sd_sigma"] = float(s_hats.std(ddof=1))

    # Xi bias
    xi_hat_stack = np.stack([met["xi_hat"] for met in rep_metrics], axis=0)  
    xi_true = rep_metrics[0]["xi_true"]
    bias_tj = xi_hat_stack.mean(axis=0) - xi_true
    sd_tj = xi_hat_stack.std(axis=0, ddof=1)
    out["avg_abs_bias_xi_jt"] = float(np.abs(bias_tj).mean())
    out["avg_sd_xi_jt"] = float(sd_tj.mean())

    return out



def _compute_blp_metrics_one_rep(blp_out: dict, true: dict, sim: SimParams):
    """
    blp_out: output dict from blp_estimator_fast 
             must contain keys: "theta", "xi", "xi_bar"
    true: must contain "xi_star" (T,J)
    sim: provides beta_mean, sigma_beta, xi_bar (scalar truth)

    Returns a dict 
    """
    theta = np.asarray(blp_out["theta"]).reshape(-1)  # (3,) [beta_p, beta_w, sigma_p]
    xi_hat = np.asarray(blp_out["xi"])               # (T,J) inside only
    xibar_hat_t = np.asarray(blp_out["xi_bar"]).reshape(-1)  # (T,)

    met = {
        # parameter point estimates
        "beta_0_hat": float(theta[0]),
        "beta_1_hat": float(theta[1]),
        "sigma_hat": float(theta[2]),

        # xibar: store market-average estimate (scalar) for bias/sd 
        "xibar_hat_mean": float(xibar_hat_t.mean()),

        # xi objects for Lu-style abs bias/sd over (t,j)
        "xi_hat": xi_hat,
        "xi_true": np.asarray(true["xi_star"]),
    }
    return met


def _summarize_blp_across_reps(rep_metrics: list, sim: SimParams):
    """
    Aggregates BLP rep metrics 
    """
    out = {}

    # xibar (scalar truth)
    xibar_hats = np.array([m["xibar_hat_mean"] for m in rep_metrics], dtype=float)
    out["bias_xibar"] = float(xibar_hats.mean() - float(sim.xi_bar))
    out["sd_xibar"] = float(xibar_hats.std(ddof=1))

    # betas
    b0 = np.array([m["beta_0_hat"] for m in rep_metrics], dtype=float)
    b1 = np.array([m["beta_1_hat"] for m in rep_metrics], dtype=float)
    out["bias_beta_0"] = float(b0.mean() - float(sim.beta_mean[0]))
    out["sd_beta_0"] = float(b0.std(ddof=1))
    out["bias_beta_1"] = float(b1.mean() - float(sim.beta_mean[1]))
    out["sd_beta_1"] = float(b1.std(ddof=1))

    # sigma (target is first random coef, for D=2 compat)
    s = np.array([m["sigma_hat"] for m in rep_metrics], dtype=float)
    target_sigma = float(sim.sigma_beta[0]) if sim.sigma_beta is not None else 0.0
    out["bias_sigma"] = float(s.mean() - target_sigma)
    out["sd_sigma"] = float(s.std(ddof=1))

    # xi_{t,j} Lu-style abs bias/sd
    xi_hat_stack = np.stack([m["xi_hat"] for m in rep_metrics], axis=0)   # (n_rep,T,J)
    xi_true_stack = np.stack([m["xi_true"] for m in rep_metrics], axis=0) # (n_rep,T,J)

    bias_tj = xi_hat_stack.mean(axis=0) - xi_true_stack.mean(axis=0)      # (T,J)
    sd_tj = xi_hat_stack.std(axis=0, ddof=1)                              # (T,J)
    out["avg_abs_bias_xi_jt"] = float(np.abs(bias_tj).mean())
    out["avg_sd_xi_jt"] = float(sd_tj.mean())

    return out


  

def plot_mcmc_and_probs(draws, true, sim, blp_iv1=None, blp_iv2=None, blp_iv3=None,
                        blp1_xibar=None, blp2_xibar=None, blp3_xibar=None, out_path_prefix=None):

    beta_draws   = draws["beta_bar"]                   # (N, D)
    sigma_draws  = np.exp(draws["r_vec"])              # (N, n_random)
    phi_mean     = draws["phi"].mean(axis=0)           # (T,)
    xibar_draws  = draws["xi_bar"]                     # (N, T)
    xibar_trace  = xibar_draws.mean(axis=1)            # (N,)  mean over markets per draw
    eta_draws    = draws["eta"]                        # (N, T, J)
    eta_trace    = eta_draws.mean(axis=(1, 2))         # (N,)  mean over T and J per draw

    D        = sim.D
    n_random = sigma_draws.shape[1]
    n_rows   = D + n_random + 4   # beta + sigma + eta + xibar + phi/gamma

    # ── pre-compute BLP and true scalars ─────────────────────────────────────
    true_xibar       = float(sim.xi_bar)
    true_eta         = float(np.mean(true["eta_star"]))          
    blp1_xibar_mean  = float(np.mean(blp1_xibar)) if blp1_xibar is not None else None
    blp2_xibar_mean  = float(np.mean(blp2_xibar)) if blp2_xibar is not None else None
    blp3_xibar_mean  = float(np.mean(blp3_xibar)) if blp3_xibar is not None else None

    fig, axes = plt.subplots(n_rows, 2, figsize=(14, 4 * n_rows))

    # ── Beta rows ─────────────────────────────────────────────────────────────
    for d in range(D):
        b_trace = beta_draws[:, d]
        true_b  = sim.beta_mean[d]

        ax = axes[d, 0]
        ax.plot(b_trace, alpha=0.7)
        ax.axhline(true_b,         color='r', ls='--', label='True')
        ax.axhline(b_trace.mean(), color='k', ls='-',  label='Posterior Mean')
        if blp_iv1 is not None: ax.axhline(blp_iv1[d], color='b',      ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axhline(blp_iv2[d], color='purple',  ls=':', label='BLP2')
        if blp_iv3 is not None: ax.axhline(blp_iv3[d], color='orange',  ls=':', label='BLP3')
        ax.set_title(f"Beta {d} Samples"); ax.legend()

        ax = axes[d, 1]
        ax.hist(b_trace, bins=30, alpha=0.6)
        ax.axvline(true_b,         color='r', ls='--', label='True')
        ax.axvline(b_trace.mean(), color='k', ls='-',  label='Posterior Mean')
        if blp_iv1 is not None: ax.axvline(blp_iv1[d], color='b',      ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axvline(blp_iv2[d], color='purple',  ls=':', label='BLP2')
        if blp_iv3 is not None: ax.axvline(blp_iv3[d], color='orange',  ls=':', label='BLP3')
        ax.set_title(f"Beta {d} Posterior"); ax.legend()

    # ── Sigma rows ────────────────────────────────────────────────────────────
    offset = D
    for k in range(n_random):
        s_trace = sigma_draws[:, k]
        true_s  = sim.sigma_beta[sim.sigma_beta > 1e-6][k] \
                  if k < np.sum(sim.sigma_beta > 1e-6) else 0.0
        row = D + k

        ax = axes[row, 0]
        ax.plot(s_trace, alpha=0.7)
        ax.axhline(true_s,         color='r', ls='--', label='True')
        ax.axhline(s_trace.mean(), color='k', ls='-',  label='Posterior Mean')
        if blp_iv1 is not None: ax.axhline(blp_iv1[offset+k], color='b',     ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axhline(blp_iv2[offset+k], color='purple', ls=':', label='BLP2')
        if blp_iv3 is not None: ax.axhline(blp_iv3[offset+k], color='orange',  ls=':', label='BLP3')
        ax.set_title(f"Sigma {k} Samples"); ax.legend()

        ax = axes[row, 1]
        ax.hist(s_trace, bins=30, alpha=0.6)
        ax.axvline(true_s,         color='r', ls='--', label='True')
        ax.axvline(s_trace.mean(), color='k', ls='-',  label='Posterior Mean')
        if blp_iv1 is not None: ax.axvline(blp_iv1[offset+k], color='b',     ls=':', label='BLP1')
        if blp_iv2 is not None: ax.axvline(blp_iv2[offset+k], color='purple', ls=':', label='BLP2')
        if blp_iv3 is not None: ax.axvline(blp_iv3[offset+k], color='orange',  ls=':', label='BLP3')
        ax.set_title(f"Sigma {k} Posterior"); ax.legend()

    # ── Eta row ───────────────────────────────────────────────────────────────
    # eta_trace shape: (N,)  — mean over T markets and J products per draw
    eta_row = D + n_random

    ax = axes[eta_row, 0]
    ax.plot(eta_trace, alpha=0.7)
    ax.axhline(true_eta,          color='r', ls='--', label='True (mean over T,J)')
    ax.axhline(eta_trace.mean(),  color='k', ls='-',  label='Posterior Mean')
    ax.set_title("eta (mean over T, J) Samples")
    ax.legend()

    ax = axes[eta_row, 1]
    ax.hist(eta_trace, bins=30, alpha=0.6)
    ax.axvline(true_eta,         color='r', ls='--', label='True (mean over T,J)')
    ax.axvline(eta_trace.mean(), color='k', ls='-',  label='Posterior Mean')
    ax.set_title("eta (mean over T, J) Posterior")
    ax.legend()

    # ── xi_bar row ────────────────────────────────────────────────────────────
    # xibar_trace shape: (N,)  — mean over T markets per draw
    xibar_row = D + n_random + 1

    ax = axes[xibar_row, 0]
    ax.plot(xibar_trace, alpha=0.7)
    ax.axhline(true_xibar,          color='r', ls='--', label='True (mean over T)')
    ax.axhline(xibar_trace.mean(),  color='k', ls='-',  label='Posterior Mean')
    if blp1_xibar_mean is not None: ax.axhline(blp1_xibar_mean, color='b',     ls=':', label='BLP1')
    if blp2_xibar_mean is not None: ax.axhline(blp2_xibar_mean, color='purple', ls=':', label='BLP2')
    if blp3_xibar_mean is not None: ax.axhline(blp3_xibar_mean, color='orange', ls=':', label='BLP3')
    ax.set_title("xi_bar (mean over T) Samples")
    ax.legend()

    ax = axes[xibar_row, 1]
    ax.hist(xibar_trace, bins=30, alpha=0.6)
    ax.axvline(true_xibar,         color='r', ls='--', label='True (mean over T)')
    ax.axvline(xibar_trace.mean(), color='k', ls='-',  label='Posterior Mean')
    if blp1_xibar_mean is not None: ax.axvline(blp1_xibar_mean, color='b',     ls=':', label='BLP1')
    if blp2_xibar_mean is not None: ax.axvline(blp2_xibar_mean, color='purple', ls=':', label='BLP2')
    if blp3_xibar_mean is not None: ax.axvline(blp3_xibar_mean, color='orange', ls=':', label='BLP3')
    ax.set_title("xi_bar (mean over T) Posterior")
    ax.legend()

    # ── Phi row ───────────────────────────────────────────────────────────────
    phi_row = D + n_random + 2

    ax = axes[phi_row, 0]
    ax.plot(draws["phi"].mean(axis=(1,)), alpha=0.7)   # (N,) — mean phi over markets per draw
    ax.set_title("Phi Mean (trace over draws)")
    ax.set_xlabel("Draw")

    ax = axes[phi_row, 1]
    ax.bar(range(len(phi_mean)), phi_mean, alpha=0.7)
    ax.set_title("Phi Mean per Market")
    ax.set_xlabel("Market t")
    ax.set_ylabel("Mean phi_t")

    # ── Gamma row ─────────────────────────────────────────────────────────────
    last_row = D + n_random + 3

    gamma_prob = draws["gamma"].mean(axis=0).ravel()    # (T*J,)
    ax = axes[last_row, 0]
    ax.plot(draws["gamma"].mean(axis=(1, 2)), alpha=0.7)  # (N,) — mean gamma per draw
    ax.set_title("Gamma Mean (trace over draws)")
    ax.set_xlabel("Draw")

    ax = axes[last_row, 1]
    ax.hist(gamma_prob, bins=30, range=(0, 1))
    ax.set_title("Gamma Inclusion Probs (all T×J)")
    ax.set_xlabel("Inclusion probability")

    plt.tight_layout()
    if out_path_prefix:
        plt.savefig(out_path_prefix + "_mcmc.png")
        plt.close(fig)
    else:
        plt.show()

# =====================================
# MAIN RUNNER
# =====================================

def run_global_grid(
    n_rep=10,
    Ts=(25, 100),
    Js=(5, 15),
    DGPs=(1, 2),
    Nt=1000,
    base_seed=123,
    out_dir=os.path.join(project_root, 'Experiments', 'Lu DGP results'),
    N_draw=300,
    blp_seed=111,
    mcmc_template=None,
    calib_template=None,
    D=2,
    save_pickles=True, 
    make_plots=True, 
    plot_every_rep=False
):
    os.makedirs(out_dir, exist_ok=True)

    # Default Config for D=2 (Original Paper Simulation Replication) 
    if D == 2:
        beta_mean = np.array([-1.0, 0.5])
        sigma_beta = np.array([1.5, 0.0])
        random_mask = np.array([1.0, 0.0])
    else:
        beta_mean = np.random.randn(D)
        sigma_beta = np.zeros(D); sigma_beta[0]=1.0
        random_mask = np.zeros(D); random_mask[0]=1.0

    print(f"True beta: {beta_mean}")
    print(f"True sigma (std): {sigma_beta[0]}")
    print(f"True xi_bar: {SimParams.xi_bar}")

    if mcmc_template is None:
        mcmc_template = MCMCParams(R0=200, G=10000, burn=5000, D=D, random_coef_mask=random_mask)



    for dgp in DGPs:
        rows = []
        for J in Js:
            for T in Ts:
                print(f"Processing DGP={dgp}, T={T}, J={J}...")
                blp1_rep, blp2_rep, blp3_rep, lu_rep, rep_out = [], [], [], [], []


                for rep in range(n_rep):
                    seed = base_seed + 100000 * dgp + 1000 * T + 10 * J + rep
                    sim = SimParams(T=T, J=J, Nt=Nt, D=D, beta_mean=beta_mean, sigma_beta=sigma_beta, seed=seed)
                    data, dataset_cl, true = generate_data_tf(dgp, sim)

                
                    # --- BLP
                    blp_out1, blp_out2, blp_out3 = _blp_estimate_one(data, true, N_draw=N_draw, seed=blp_seed + seed)
                    
                    th_iv1 = np.asarray(blp_out1["theta"]).reshape(-1)
                    th_iv2 = np.asarray(blp_out2["theta"]).reshape(-1)
                    th_iv3 = np.asarray(blp_out3["theta"]).reshape(-1)
                    
                    blp1_rep.append(_compute_blp_metrics_one_rep(blp_out1, true, sim))
                    blp2_rep.append(_compute_blp_metrics_one_rep(blp_out2, true, sim))
                    blp3_rep.append(_compute_blp_metrics_one_rep(blp_out3, true, sim))


                    # --- Lu2025
                    mcmc = MCMCParams(**vars(mcmc_template))
                    draws, _, _ = run_choice_learn_mcmc(dataset_cl, sim, mcmc, verbose=True,)
                    


                    metric = _compute_metrics_one_rep(draws, true, sim)
                    lu_rep.append(metric)

                    # Store result
                    res_dict = {"rep": rep, "DGP": dgp, "T": T, "J": J}
                    
                    # Flatten metrics
                    for k, v in metric.items():
                        if isinstance(v, (float, int)): res_dict["lu_"+k] = v             

                    for k_idx, name in enumerate(["beta_0", "beta_1", "sigma"][:len(th_iv1)]):
                        res_dict[f"blp1_{name}"] = float(th_iv1[k_idx])
                        res_dict[f"blp2_{name}"] = float(th_iv2[k_idx])
                        res_dict[f"blp3_{name}"] = float(th_iv3[k_idx])

                    res_dict["blp1_xibar_hat_mean"] = blp1_rep[-1]["xibar_hat_mean"]
                    res_dict["blp2_xibar_hat_mean"] = blp2_rep[-1]["xibar_hat_mean"]
                    res_dict["blp3_xibar_hat_mean"] = blp3_rep[-1]["xibar_hat_mean"]


                    rep_out.append(res_dict)

                    # Save Pickles
                    if save_pickles:
                        pkl_path = os.path.join(out_dir, f"full_DGP{dgp}_T{T}_J{J}_rep{rep}.pkl")
                        with open(pkl_path, "wb") as f:
                            pickle.dump({"draws": draws, "blp1": blp_out1, "blp2": blp_out2, "blp3": blp_out3}, f)

                    # Plot
                    if make_plots and (plot_every_rep or rep == 0):
                        prefix = os.path.join(out_dir, f"plot_DGP{dgp}_T{T}_J{J}_rep{rep}")
                        plot_mcmc_and_probs(
                            draws, true, sim,
                            blp_iv1=th_iv1, blp_iv2=th_iv2, blp_iv3=th_iv3,
                            blp1_xibar=np.asarray(blp_out1["xi_bar"]).reshape(-1),
                            blp2_xibar=np.asarray(blp_out2["xi_bar"]).reshape(-1),
                            blp3_xibar=np.asarray(blp_out3["xi_bar"]).reshape(-1),
                            out_path_prefix=prefix
                        )

                # --- Summarize
                # Lu summary
                lu_summary = _summarize_across_reps(lu_rep, sim)

                # BLP summaries
                blp1_summary = _summarize_blp_across_reps(blp1_rep, sim)
                blp2_summary = _summarize_blp_across_reps(blp2_rep, sim)
                blp3_summary = _summarize_blp_across_reps(blp3_rep, sim)



                row = {
                    "DGP": dgp, "T": T, "J": J,
                
                    # Lu
                    "lu_bias_xibar": lu_summary["bias_xibar"],
                    "lu_sd_xibar": lu_summary["sd_xibar"],
                    "lu_bias_beta_0": lu_summary["bias_beta_0"],
                    "lu_sd_beta_0": lu_summary["sd_beta_0"],
                    "lu_bias_beta_1": lu_summary["bias_beta_1"],
                    "lu_sd_beta_1": lu_summary["sd_beta_1"],
                    "lu_bias_sigma_beta_0": lu_summary["bias_sigma"],
                    "lu_sd_sigma_beta_0": lu_summary["sd_sigma"],
                    "lu_avg_abs_bias_xi_jt": lu_summary["avg_abs_bias_xi_jt"],
                    "lu_avg_sd_xi_jt": lu_summary["avg_sd_xi_jt"],
                
                    # BLP1
                    "blp1_bias_xibar": blp1_summary["bias_xibar"],
                    "blp1_sd_xibar": blp1_summary["sd_xibar"],
                    "blp1_bias_beta_0": blp1_summary["bias_beta_0"],
                    "blp1_sd_beta_0": blp1_summary["sd_beta_0"],
                    "blp1_bias_beta_1": blp1_summary["bias_beta_1"],
                    "blp1_sd_beta_1": blp1_summary["sd_beta_1"],
                    "blp1_bias_sigma_beta_0": blp1_summary["bias_sigma"],
                    "blp1_sd_sigma_beta_0": blp1_summary["sd_sigma"],
                    "blp1_avg_abs_bias_xi_jt": blp1_summary["avg_abs_bias_xi_jt"],
                    "blp1_avg_sd_xi_jt": blp1_summary["avg_sd_xi_jt"],
                
                    # BLP2
                    "blp2_bias_xibar": blp2_summary["bias_xibar"],
                    "blp2_sd_xibar": blp2_summary["sd_xibar"],
                    "blp2_bias_beta_0": blp2_summary["bias_beta_0"],
                    "blp2_sd_beta_0": blp2_summary["sd_beta_0"],
                    "blp2_bias_beta_1": blp2_summary["bias_beta_1"],
                    "blp2_sd_beta_1": blp2_summary["sd_beta_1"],
                    "blp2_bias_sigma_beta_0": blp2_summary["bias_sigma"],
                    "blp2_sd_sigma_beta_0": blp2_summary["sd_sigma"],
                    "blp2_avg_abs_bias_xi_jt": blp2_summary["avg_abs_bias_xi_jt"],
                    "blp2_avg_sd_xi_jt": blp2_summary["avg_sd_xi_jt"],

                    # BLP3
                    "blp3_bias_xibar": blp3_summary["bias_xibar"],
                    "blp3_sd_xibar": blp3_summary["sd_xibar"],
                    "blp3_bias_beta_0": blp3_summary["bias_beta_0"],
                    "blp3_sd_beta_0": blp3_summary["sd_beta_0"],
                    "blp3_bias_beta_1": blp3_summary["bias_beta_1"],
                    "blp3_sd_beta_1": blp3_summary["sd_beta_1"],
                    "blp3_bias_sigma_beta_0": blp3_summary["bias_sigma"],
                    "blp3_sd_sigma_beta_0": blp3_summary["sd_sigma"],
                    "blp3_avg_abs_bias_xi_jt": blp3_summary["avg_abs_bias_xi_jt"],
                    "blp3_avg_sd_xi_jt": blp3_summary["avg_sd_xi_jt"],
                }

    
                rows.append(row)

                # Save Reps
                pd.DataFrame(rep_out).to_csv(os.path.join(out_dir, f"reps_DGP{dgp}_T{T}_J{J}.csv"), index=False)

        # Save Global Table
        pd.DataFrame(rows).to_csv(os.path.join(out_dir, f"table_DGP{dgp}.csv"), index=False)
        print(f"Saved table for DGP {dgp}")



In [6]:
run_global_grid(n_rep=10,
        DGPs=(1, 2, 3, 4),
        Ts=(25, 100),
        Js=(5, 15),
        D=2)

True beta: [-1.   0.5]
True sigma (std): 1.5
True xi_bar: -1.0
Processing DGP=1, T=25, J=5...

--- Starting BLP IV Type 1 ---


Stage 1 (IV=1): 32it [00:00, 406.41it/s, sigma=1.5000, obj=1.3668e-02]
Stage 2 (IV=1): 30it [00:00, 436.13it/s, sigma=1.5000, obj=3.4416e-02]


BLP 1 final beta: [-0.83307627  0.28904495]
BLP 1 final sigma (std): 1.5000046246362952

--- Starting BLP IV Type 2 ---


Stage 1 (IV=2): 38it [00:00, 431.74it/s, sigma=1.5000, obj=6.2228e-02]
Stage 2 (IV=2): 68it [00:00, 437.31it/s, sigma=1.4970, obj=5.2627e-02]


BLP 2 final beta: [-2.38757935  0.23958186]
BLP 2 final sigma (std): 1.4970066853004624

--- Starting BLP IV Type 3 ---


...

Stage 1 (IV=3): 2it [00:00, 84.45it/s, sigma=1.5000, obj=1.0013e-25]
Stage 2 (IV=3): 2it [00:00, 65.24it/s, sigma=1.5000, obj=8.8827e-26]


BLP 3 final beta: [-0.19488118  0.51546269]
BLP 3 final sigma (std): 1.5

--- Starting Bayesian Shrinkage Random Logit ---


MCMC chain: 100%|█████████████████████████████████████████████████████████████████| 10000/10000 [03:38<00:00, 45.72it/s]


Final beta:  [-0.93655323  0.47480576]
Final sigma: [1.45847777]
Accept  beta=0.304  r=0.310  xi=0.289  eta=0.302  phi=0.325  gamma=1 (Gibbs)
Step sizes:
  step_beta = 0.011783
  step_r = 0.022396
  step_xi = 0.500000
  step_eta = 0.450000
  step_phi = 0.104492
Saved table for DGP 4


/var/folders/z9/wlmv5z6x1h96wq3x7djmx_2w0000gn/T/ipykernel_61374/630365398.py:58: RuntimeWarning: Mean of empty slice
  "pr_gamma_1_eta_z": float(np.nanmean([met["pr_gamma_1_eta_z"] for met in rep_metrics])),
